# SomaliTTS — تنزيل شامل للموارد الصومالية (تشغيل مرة واحدة)

هذا الدفتر ينزّل Hubi-TTS وبيانات Som TTS وAyaan إلى مجلد واحد داخل Kaggle، ويحفظ تقريرًا بالملفات والأحجام. اختر GPU وفعّل Internet ثم شغّل الخلية الوحيدة. لا يشغّل تدريبًا ولا يغيّر Laravel. بعد اكتماله استخدم Save Version مع حفظ المخرجات، ثم شارك رابط نسخة Kaggle أو حمّل مجلد النتائج/ملفاته حتى أراجعها وأجهّز الربط.

In [ ]:
import os, sys, json, time, traceback
from pathlib import Path
from datetime import datetime, timezone

# Install the downloader library once in this Kaggle session.
!pip -q install -U huggingface_hub
from huggingface_hub import snapshot_download

ROOT = Path('/kaggle/working/SomaliTTS-assets')
ROOT.mkdir(parents=True, exist_ok=True)
HF_TOKEN = os.environ.get('HF_TOKEN')  # optional; do not paste secrets into notebook cells

# Deliberately excludes MMS Somali because its published license is non-commercial.
ASSETS = [
    {'repo_id': 'ayeeyokoris/hubi-tts', 'repo_type': 'model', 'folder': 'models/hubi-tts', 'purpose': 'experimental Somali voice-cloning model; license not clearly declared'},
    {'repo_id': 'zakihassan/som_tts', 'repo_type': 'dataset', 'folder': 'datasets/som_tts', 'purpose': 'Somali TTS dataset; license metadata needs reconciliation'},
    {'repo_id': 'maanka2/ayaan-somali-tts-corpus', 'repo_type': 'dataset', 'folder': 'datasets/ayaan-somali-tts-corpus', 'purpose': 'Somali speech corpus; review speaker-recording permissions'},
]

manifest = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'root': str(ROOT),
    'assets': [],
    'notice': 'Downloads retain upstream terms. This is not a license clearance or proof of commercial rights.'
}

for item in ASSETS:
    target = ROOT / item['folder']
    target.mkdir(parents=True, exist_ok=True)
    record = dict(item)
    print('\n' + '='*72 + '\nDownloading: ' + item['repo_id'], flush=True)
    try:
        downloaded = snapshot_download(
            repo_id=item['repo_id'],
            repo_type=item['repo_type'],
            local_dir=str(target),
            token=HF_TOKEN,
            max_workers=4,
        )
        files = [p for p in target.rglob('*') if p.is_file() and '.cache' not in p.parts]
        total_bytes = sum(p.stat().st_size for p in files)
        record.update({'status': 'downloaded', 'path': str(downloaded), 'file_count': len(files), 'bytes': total_bytes})
        print(f"DONE: {item['repo_id']} | {len(files)} files | {total_bytes/1024**3:.2f} GiB", flush=True)
    except Exception as e:
        record.update({'status': 'failed', 'error': repr(e)})
        print(f"FAILED: {item['repo_id']}: {e}", flush=True)
    manifest['assets'].append(record)
    (ROOT / 'download-manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

# Final inventory: file paths, sizes, and SHA-256 hashes so the received bundle can be checked.
import hashlib
inventory = []
for p in ROOT.rglob('*'):
    if p.is_file() and '.cache' not in p.parts:
        h = hashlib.sha256()
        with p.open('rb') as f:
            for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
                h.update(chunk)
        inventory.append({'path': str(p.relative_to(ROOT)), 'bytes': p.stat().st_size, 'sha256': h.hexdigest()})
(ROOT / 'file-inventory.json').write_text(json.dumps(inventory, ensure_ascii=False, indent=2), encoding='utf-8')
manifest['total_files'] = len(inventory)
manifest['total_bytes'] = sum(x['bytes'] for x in inventory)
manifest['finished_at'] = datetime.now(timezone.utc).isoformat()
(ROOT / 'download-manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

print('\n' + '='*72)
print('SOMALITTS DOWNLOAD FINISHED')
print('Output folder:', ROOT)
print('Total files:', manifest['total_files'])
print('Total size: %.2f GiB' % (manifest['total_bytes']/1024**3))
print('Manifest:', ROOT / 'download-manifest.json')
print('Inventory:', ROOT / 'file-inventory.json')
print('\nIf an asset failed, its exact error is recorded in download-manifest.json. Save this notebook version with outputs enabled.')
